<a href="https://colab.research.google.com/github/tschwider09/cross-species-ephys-transfer/blob/main/Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dependencies Install

In [ ]:
!pip install allensdk --no-deps

# Then install its required dependencies manually
!pip install requests h5py pandas matplotlib scipy numpy
!pip install SimpleITK


In [ ]:

!pip install "numpy>=1.21,<2.0" --force-reinstall
!pip install methodtools
!pip install pynwb


In [ ]:
!pip install dandi

In [ ]:
!pip install ipfx

In [4]:
import numpy as np
from pathlib import Path
from ipfx.dataset.create import create_ephys_data_set
from ipfx.utilities import drop_failed_sweeps
from ipfx.epochs import get_stim_epoch
import ipfx.stimulus_protocol_analysis as spa
import ipfx.feature_vectors as fv
import ipfx.error as er
from ipfx.feature_extractor import SpikeFeatureExtractor, SpikeTrainFeatureExtractor
import glob
import os
import pandas as pd
from dandi.dandiapi import DandiAPIClient
from google.colab import files
import json
from collections import Counter
import re


##Ephys Data Downloading

In [ ]:
#pulling the files from DANDI

DANDISET_ID = "000636"    #use 000020 for the mouse files
DOWNLOAD_DIR = "nwb_data"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

client = DandiAPIClient()
dandiset = client.get_dandiset(DANDISET_ID)
assets = list(dandiset.get_assets())

for asset in assets:
    if asset.path.endswith(".nwb"):

## Feature Extration

In [5]:
def _stim_window(sweep_set):
    s0 = sweep_set.sweeps[0]
    start_idx, end_idx = get_stim_epoch(s0.i)
    return s0.t[start_idx], s0.t[end_idx]

def safe_subsample_average(x, width):
    # drop the last len(x) % width samples
    n = len(x) // width
    if n == 0:
        return np.array([])      # nothing to average
    x2 = x[: n * width]
    return np.nanmean(x2.reshape(n, width), axis=1)

fv._subsample_average = safe_subsample_average

def _safe_analyze(analysis, sweep_set, name):
    """
    Run IPFX analysis for a protocol, but don't crash if there are no spiking sweeps.
    Returns (features_dict or None, error_message or None).
    """
    if sweep_set is None or len(sweep_set.sweeps) == 0:
        return None, f"{name}: no sweeps"
    try:
        feats = analysis.analyze(sweep_set)
        return feats, None
    except er.FeatureError as e:
        return None, f"{name}: {e}"
    except Exception as e:
        return None, f"{name}: {type(e).__name__}: {e}"


def _first_spiking_index(spiking_df):
    # pick lowest-amp spiking sweep (ties resolved by order)
    return spiking_df.sort_values("stim_amp").index[0]


def make_gouwens_feature_vectors(nwb_file):
    """
    Returns a dict of the 12 Gouwens vectors.
    Any vector that cannot be computed (missing protocol/no spikes) is set to None.
    Keys:
      first_ap_v, first_ap_dv, isi_shape, inst_freq, spiking_threshold_v,
      spiking_peak_v, spiking_width, spiking_fast_trough_v,
      spiking_upstroke_downstroke_ratio, step_subthresh, subthresh_norm, psth
    """
    ds = create_ephys_data_set(nwb_file=nwb_file)
    drop_failed_sweeps(ds)

    # ---- Gather sweeps by protocol ----
    lsq_tbl = ds.filtered_sweep_table(stimuli=ds.ontology.long_square_names)
    ssq_tbl = ds.filtered_sweep_table(stimuli=ds.ontology.short_square_names)
    ramp_tbl = ds.filtered_sweep_table(stimuli=ds.ontology.ramp_names)

    lsq = ds.sweep_set(lsq_tbl.sweep_number) if len(lsq_tbl) else None
    ssq = ds.sweep_set(ssq_tbl.sweep_number) if len(ssq_tbl) else None
    ramp = ds.sweep_set(ramp_tbl.sweep_number) if len(ramp_tbl) else None

    for S in (lsq, ssq, ramp):
        if S is None or len(S.sweeps) == 0:
            continue
        S.select_epoch("recording")
        S.align_to_start_of_epoch("experiment")

    lsq_start, lsq_end = (_stim_window(lsq) if lsq and len(lsq.sweeps) else (None, None))
    ssq_start, ssq_end = (_stim_window(ssq) if ssq and len(ssq.sweeps) else (None, None))
    ramp_start, ramp_end = (_stim_window(ramp) if ramp and len(ramp.sweeps) else (None, None))

    # ---- Run protocol analyses safely ----
    # Long square
    spx_lsq = SpikeFeatureExtractor(start=lsq_start, end=lsq_end) if lsq_start is not None else None
    sptx_lsq = SpikeTrainFeatureExtractor(start=lsq_start, end=lsq_end) if lsq_start is not None else None
    lsq_feats, lsq_err = _safe_analyze(
        spa.LongSquareAnalysis(spx_lsq, sptx_lsq, subthresh_min_amp=-100.0)
        if spx_lsq is not None else spa.LongSquareAnalysis(None, None),
        lsq, "LongSquare"
    )

    # Short square
    spx_ssq = SpikeFeatureExtractor(start=ssq_start, end=ssq_end) if ssq_start is not None else None
    sptx_ssq = SpikeTrainFeatureExtractor(start=ssq_start, end=ssq_end) if ssq_start is not None else None
    ssq_feats, ssq_err = _safe_analyze(
        spa.ShortSquareAnalysis(spx_ssq, sptx_ssq) if spx_ssq is not None else spa.ShortSquareAnalysis(None, None),
        ssq, "ShortSquare"
    )

    # Ramp
    spx_ramp = SpikeFeatureExtractor(start=ramp_start, end=None) if ramp_start is not None else None
    sptx_ramp = SpikeTrainFeatureExtractor(start=ramp_start, end=None) if ramp_start is not None else None
    ramp_feats, ramp_err = _safe_analyze(
        spa.RampAnalysis(spx_ramp, sptx_ramp) if spx_ramp is not None else spa.RampAnalysis(None, None),
        ramp, "Ramp"
    )

    out = {k: None for k in [
        "first_ap_v", "first_ap_dv", "isi_shape", "inst_freq",
        "spiking_threshold_v", "spiking_peak_v", "spiking_width",
        "spiking_fast_trough_v", "spiking_upstroke_downstroke_ratio",
        "step_subthresh", "subthresh_norm", "psth"
    ]}

    # ---- First AP vectors (concatenate whatever protocols have a first AP) ----
    sweeps_for_first_ap, spikes_for_first_ap = [], []

    # Short square: lowest-amp spiking sweep
    if ssq_feats is not None and "spiking_sweeps" in ssq_feats and len(ssq_feats["spiking_sweeps"]) > 0:
        try:
            ssq_idx = _first_spiking_index(ssq_feats["spiking_sweeps"])
            sweeps_for_first_ap.append(ssq.sweeps[ssq_idx])
            spikes_for_first_ap.append(ssq_feats["spikes_set"][ssq_idx])
        except Exception:
            pass  # keep going

    # Long square: rheobase sweep
    if lsq_feats is not None and "rheobase_sweep" in lsq_feats and lsq_feats["rheobase_sweep"] is not None:
        try:
            lsq_idx = lsq_feats["rheobase_sweep"].name
            sweeps_for_first_ap.append(lsq.sweeps[lsq_idx])
            spikes_for_first_ap.append(lsq_feats["spikes_set"][lsq_idx])
        except Exception:
            pass

    # Ramp: first spiking sweep
    if ramp_feats is not None and "spiking_sweeps" in ramp_feats and len(ramp_feats["spiking_sweeps"]) > 0:
        try:
            ramp_idx = _first_spiking_index(ramp_feats["spiking_sweeps"])
            sweeps_for_first_ap.append(ramp.sweeps[ramp_idx])
            spikes_for_first_ap.append(ramp_feats["spikes_set"][ramp_idx])
        except Exception:
            pass

    if len(sweeps_for_first_ap) > 0:
        ap_v, ap_dv = fv.first_ap_vectors(
            sweeps_for_first_ap, spikes_for_first_ap,
            target_sampling_rate=50_000,  # 50 kHz
            window_length=0.003            # 3 ms post-threshold
        )
        out["first_ap_v"] = ap_v
        out["first_ap_dv"] = ap_dv
    # else: stay None if no protocol had a spiking sweep

    # ---- Long-square–based vectors (require LSQ with spikes) ----
    if lsq_feats is not None and "rheobase_sweep" in lsq_feats and lsq_feats["rheobase_sweep"] is not None:
        # suprathreshold bins at rheobase, +40 pA, +80 pA (tolerance 5 pA)
        supra_info = fv.identify_suprathreshold_spike_info(
            lsq_feats, target_amplitudes=np.array([0, 40, 80]), amp_tolerance=5
        )

        # (3) ISI shape (needs a decent spike train)
        try:
            sel_sweep, sel_spike_info = fv.identify_sweep_for_isi_shape(
                lsq, lsq_feats, duration=(lsq_end - lsq_start), min_spike=5
            )
            out["isi_shape"] = fv.isi_shape(sel_sweep, sel_spike_info, end=lsq_end, n_points=100)
        except Exception:
            pass

        # (4) Inst. freq
        try:
            out["inst_freq"] = fv.inst_freq_vector(supra_info, lsq_start, lsq_end, width=20)
        except Exception:
            pass

        # (5–9) Spike feature timecourses
        for name in [
            ("spiking_threshold_v", "threshold_v"),
            ("spiking_peak_v", "peak_v"),
            ("spiking_width", "width"),
            ("spiking_fast_trough_v", "fast_trough_v"),
            ("spiking_upstroke_downstroke_ratio", "upstroke_downstroke_ratio"),
        ]:
            try:
                out[name[0]] = fv.spike_feature_vector(name[1], supra_info, lsq_start, lsq_end, width=20)
            except Exception:
                pass

        # (10–11) Subthreshold hyperpolarizing responses (downsampled 10 ms)
        try:
          amp_sweep_dict, deflect_dict = fv.identify_subthreshold_hyperpol_with_amplitudes(lsq_feats, lsq)
          out["step_subthresh"] = fv.step_subthreshold(
                amp_sweep_dict, target_amps=[-30.0, -50.0, -90.0],
                start=lsq_start, end=lsq_end, subsample_interval=0.01
            )
          out["subthresh_norm"] = fv.subthresh_norm(
                amp_sweep_dict, deflect_dict,
                start=lsq_start, end=lsq_end, target_amp=-110.0, subsample_interval=0.01
            )
        except Exception:

            pass

        # (12) PSTH (20–50 ms bin ok; 20 ms matches others)
        try:
            out["psth"] = fv.psth_vector(supra_info, lsq_start, lsq_end, width=20)
        except Exception:
            pass

    return out

In [ ]:
nwb_path = "/content/nwb_data/sub-720619787_ses-720868659_icephys.nwb" #choose an example file an test out the dictionairy based featurization
feats = make_gouwens_feature_vectors(nwb_path)


In [ ]:
#grab file paths
folder_path = Path("/content/nwb_data")
file_paths = sorted(folder_path.glob("*.nwb"))
file_paths = [str(p) for p in file_paths]

In [ ]:
#use feats to make a template dictionairy
features_dict = feats.copy()
for i in features_dict.keys():
  features_dict[i] = []

In [ ]:
#go through the dataset
path_list = []
for path in file_paths:
  print(path)
  try:
    curr_feats = make_gouwens_feature_vectors(path)
    for i in features_dict.keys():
      features_dict[i].append(curr_feats[i])
    path_list.append(path)
  except:
    print(f"{path} caused an error")

In [ ]:
#visualize features
import seaborn as sns
for potential in features_dict["first_ap_dv"]:
  sns.lineplot(potential)


##T-Type Label Extraction

### Mice T-Type Extraction

In [ ]:
metadata = pd.read_csv("20200711_patchseq_metadata_mouse.csv", index_col=1)

In [ ]:
file_manifest = pd.read_csv("/content/2021-09-13_mouse_file_manifest.csv", index_col=1)

In [ ]:
ids = []
for path in path_list:
  ids.append(int(file_manifest.loc[path[18:]]["cell_specimen_id"]))

In [ ]:
t_types = []
for id in ids:
  t_types.append(metadata.loc[id]["T-type Label"].split(" ", 1)[0]) #we only have enoguh data to use the broad subclass label
t_types

In [ ]:
#check distribution
counts = Counter(t_types)
print(counts)

In [22]:
#here is where we clean away t_types that Gouwenns didn't considered and ar ebarely represented
def remove_by_ttypes(features_dict, y_ttypes, bad_ttypes):

    y_arr = np.array(y_ttypes, dtype=object)

    is_bad = np.isin(y_arr, bad_ttypes) | pd.isna(y_arr) | (y_arr == "nan")

    keep_mask = ~is_bad

    new_features_dict = {}
    for key, arr in features_dict.items():
        arr = np.asarray(arr)
        assert arr.shape[0] == len(y_arr), f"{key} has wrong row count"
        new_features_dict[key] = arr[keep_mask]

    # Filter labels
    new_y_ttypes = y_arr[keep_mask]

    removed = np.sum(is_bad)
    print(f"Removed {removed} samples (bad types {bad_ttypes} or NaN).")
    print(f"Remaining samples: {len(new_y_ttypes)}")

    return new_features_dict, new_y_ttypes


In [ ]:
features_dict, t_types = remove_by_ttypes(features_dict, t_types, ["Serpinf1", "L2/3","Meis2"])

### Human T-Type Extraction

In [20]:
import pandas as pd
metadata = pd.read_csv("LeeDalley_manuscript_metadata.csv")
metadata = metadata.set_index('nwb_id')


In [21]:
def get_id(path):
    m = re.search(r'(?i)(?:^|[/\\_])ses-([A-Za-z0-9]+)(?=[_./\\]|$)', path)
    return m.group(1)

In [ ]:
t_types = []
for i in path_list:
  t_types.append(metadata.loc[float(get_id(i))]["Revised_subclass_label"])
t_types

In [ ]:
#check distribution
counts = Counter(t_types)
print(counts)

## LSTM Stacking

In [ ]:
family_order = list(features_dict.keys())   # or sorted(X_human.keys())

# Coerce each family to clean 2D (N, d_i), auto-fixing ragged rows
def coerce_family_to_2d(arr_like, fill_value=0.0, name=""):


    rows = []
    lengths = []
    max_len = 0

    for row in arr_like:
        if row is None:
            r = np.array([], dtype=np.float32)
        elif np.isscalar(row):
            r = np.array([row], dtype=np.float32)
        else:
            r = np.asarray(row, dtype=np.float32).ravel()

        rows.append(r)
        lengths.append(r.size)
        if r.size > max_len:
            max_len = r.size
    # "slightly ragged" is fine; this is just a check diagnostic
    if name:
        c = Counter(lengths)
        if len(c) > 1:
            print(f"[INFO] Auto-padding ragged rows in {name}. Length counts (top 5): {c.most_common(5)}")

    # Build padded 2D array
    out = np.full((len(rows), max_len), fill_value, dtype=np.float32)
    for i, r in enumerate(rows):
        if r.size:
            out[i, :r.size] = r

    return out

# Apply coercion
for k in family_order:
    features_dict[k] = coerce_family_to_2d(features_dict[k], name=k)

# Verify consistent N
N = None
for k in family_order:
    n_k = features_dict[k].shape[0]
    if N is None:
        N = n_k
    else:
        assert n_k == N, f"Family {k} has {n_k} rows, but expected {N}."

# Inspect dims
family_dims = [features_dict[k].shape[1] for k in family_order]
D_total = int(sum(family_dims))
print("Per-family dims:", family_dims)
print("Total per-sample feature length (D_total):", D_total)

# Build flat features
human_X_flat = np.concatenate([features_dict[k] for k in family_order], axis=1)
print("X_flat shape:", human_X_flat.shape)

# LSTM prep: pad each family to same width then stack
step_dim_max = max(family_dims)

def pad_to_dim(arr, target_dim):
    N_local, d = arr.shape
    if d == target_dim:
        return arr
    pad_width = target_dim - d
    return np.pad(arr, ((0, 0), (0, pad_width)), mode="constant", constant_values=0.0)

padded_families = [pad_to_dim(features_dict[k], step_dim_max) for k in family_order]
human_X_seq = np.stack(padded_families, axis=1)
print("X_seq shape:", human_X_seq.shape)